In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

In [2]:
file_path = r"E:\R_Thesis\SAS_2024F.xlsx"

coordinates = pd.read_excel(file_path, sheet_name="Coordinates")
season_a = pd.read_excel(file_path, sheet_name="SeasonA")
season_b = pd.read_excel(file_path, sheet_name="SeasonB")
season_c = pd.read_excel(file_path, sheet_name="SeasonC")

print(season_a.shape, season_b.shape, season_c.shape)

(39330, 60) (35584, 67) (1949, 61)


In [3]:
season_all = pd.concat([season_a, season_b, season_c], ignore_index=True)
print("Combined data:", season_all.shape)

Combined data: (76863, 81)


In [5]:
data = season_all.merge(coordinates, on="District_Cl", how="left")

print("Merged data:", data.shape)
print("Missing latitude:", data["latitude"].isna().sum())

Merged data: (76863, 84)
Missing latitude: 0


In [6]:
data["total_loss"] = data["Q.2.39 Qty of this crop what is the quantity that has been lost?"]
data["gross_production"] = data["Q.2.21 Total quantity of harvest for this season (in Kg)"]

In [7]:
data["PHL_binary"] = (data["total_loss"] > 0).astype(int)

data["PHL_rate"] = data["total_loss"] / data["gross_production"]
data["PHL_rate"] = data["PHL_rate"].replace([np.inf, -np.inf], 0).fillna(0).clip(0, 1)

In [9]:
# Storage access
data["Storage_access"] = data[
    "Q.2.37 What is the storage facility used during this agricultural season?"
].apply(lambda x: 0 if str(x).lower() in ["none", "no", "nan"] else 1)

In [10]:
formal_markets = ["Market", "Cooperative", "Company", "Association"]

data["Market_access"] = data[
    "Q.2.27 On which market this crop was sold?"
].apply(lambda x: 1 if str(x) in formal_markets else 0)

In [11]:
X_core = data[[
    "1.8 Gender",
    "1.9 Age",
    "2.2 Plot area(sqm)",
    "District_Cl"
]]

In [12]:
X_practices = data[[
    "4.15 Has this plot been irrigated during this agricultural season?",
    "3.3 Have you used organic fertilizer in this plot during this season?",
    "3.9 Have you used inorganic fertilizer in this plot during this season?",
    "3.18 Did you use pesticide/Fungicide in any of your plots during this season?"
]]

In [13]:
X_farm = data[[
    "4.1 What is the degree of erosion on this plot?",
    "4.6 Is this plot located in land consolidated site in this season?"
]]

In [14]:
X = pd.concat([X_core, X_practices, X_farm], axis=1)

Y = data["PHL_binary"]
D = data["Storage_access"]
W = data["Market_access"]

In [15]:
ml_data = pd.concat([Y, D, W, X], axis=1)

print("Before drop NA:", ml_data.shape)

ml_data = ml_data.dropna()

print("After drop NA:", ml_data.shape)

Before drop NA: (76863, 13)
After drop NA: (2811, 13)


In [16]:
ml_data = pd.get_dummies(
    ml_data,
    columns=["District_Cl"],
    drop_first=True
)

In [17]:
Y = ml_data["PHL_binary"]
X = ml_data.drop(columns=["PHL_binary", "Storage_access", "Market_access"])

In [18]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, Y,
    test_size=0.2,
    random_state=42,
    stratify=Y
)

print(X_train.shape, X_test.shape)

(2248, 38) (563, 38)


In [21]:
X_train = pd.get_dummies(X_train)
X_test = pd.get_dummies(X_test)

In [22]:
X_train, X_test = X_train.align(X_test, join="left", axis=1, fill_value=0)

In [23]:
print(X_train.shape)

(2248, 47)


In [24]:
X_train.dtypes.value_counts()

bool       45
float64     2
Name: count, dtype: int64

In [25]:
X_train.select_dtypes(include="object").head()

""
1143
6486
76053
38587
39052


In [26]:
X_train.select_dtypes(include="object").columns

Index([], dtype='object')

In [27]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [30]:
X_train.select_dtypes(include="object").columns

Index([], dtype='object')

In [31]:
X_train = X_train.apply(pd.to_numeric, errors="raise")
X_test = X_test.apply(pd.to_numeric, errors="raise")

In [32]:
X_train.select_dtypes(include="object")

""
1143
6486
76053
38587
39052
...
76827
5448
4375
14991


In [34]:
print(X_train.isna().sum().sum())

0


In [36]:
type(X_train_scaled)

numpy.ndarray

In [37]:
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)

In [39]:
# Convert bool → int (0/1)
X_train = X_train.astype(int)
X_test = X_test.astype(int)

In [44]:
print(y_train.value_counts())

PHL_binary
0    2002
1     246
Name: count, dtype: int64


In [43]:
from sklearn.linear_model import LogisticRegression

logit = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

logit.fit(X_train_scaled, y_train)

y_pred_logit = logit.predict(X_test_scaled)

In [45]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

logit = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

logit.fit(X_train_scaled, y_train)

y_pred_logit = logit.predict(X_test_scaled)

print("Accuracy:", accuracy_score(y_test, y_pred_logit))
print(confusion_matrix(y_test, y_pred_logit))
print(classification_report(y_test, y_pred_logit))

Accuracy: 0.6181172291296625
[[302 200]
 [ 15  46]]
              precision    recall  f1-score   support

           0       0.95      0.60      0.74       502
           1       0.19      0.75      0.30        61

    accuracy                           0.62       563
   macro avg       0.57      0.68      0.52       563
weighted avg       0.87      0.62      0.69       563



In [48]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# 1. Define Ridge Logistic Regression (L2 penalty)
ridge = LogisticRegression(
    penalty='l2',
    class_weight='balanced',
    max_iter=1000
)

# 2. Train the model
ridge.fit(X_train_scaled, y_train)

# 3. Predict
y_pred_ridge = ridge.predict(X_test_scaled)

# 4. Results
print("=== RIDGE LOGISTIC REGRESSION RESULTS ===")
print("Accuracy:", accuracy_score(y_test, y_pred_ridge))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_ridge))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_ridge))

=== RIDGE LOGISTIC REGRESSION RESULTS ===
Accuracy: 0.6181172291296625

Confusion Matrix:
[[302 200]
 [ 15  46]]

Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.60      0.74       502
           1       0.19      0.75      0.30        61

    accuracy                           0.62       563
   macro avg       0.57      0.68      0.52       563
weighted avg       0.87      0.62      0.69       563



In [49]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

lasso = LogisticRegression(
    penalty='l1',
    solver='liblinear',
    class_weight='balanced',
    max_iter=1000
)

lasso.fit(X_train_scaled, y_train)

y_pred_lasso = lasso.predict(X_test_scaled)

print("=== LASSO LOGISTIC REGRESSION RESULTS ===")
print("Accuracy:", accuracy_score(y_test, y_pred_lasso))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_lasso))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_lasso))

=== LASSO LOGISTIC REGRESSION RESULTS ===
Accuracy: 0.6181172291296625

Confusion Matrix:
[[302 200]
 [ 15  46]]

Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.60      0.74       502
           1       0.19      0.75      0.30        61

    accuracy                           0.62       563
   macro avg       0.57      0.68      0.52       563
weighted avg       0.87      0.62      0.69       563



In [51]:
from sklearn.linear_model import LogisticRegression

lasso = LogisticRegression(
    penalty='l1',
    solver='liblinear',
    C=0.5,
    class_weight='balanced'
)

lasso.fit(X_train_scaled, y_train)

y_pred = lasso.predict(X_test_scaled)

In [52]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

print("=== BALANCED LASSO RESULTS ===")
print("Accuracy:", accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

=== BALANCED LASSO RESULTS ===
Accuracy: 0.6198934280639432
[[302 200]
 [ 14  47]]
              precision    recall  f1-score   support

           0       0.96      0.60      0.74       502
           1       0.19      0.77      0.31        61

    accuracy                           0.62       563
   macro avg       0.57      0.69      0.52       563
weighted avg       0.87      0.62      0.69       563



In [59]:
print(ml_data.columns)

Index(['PHL_binary', 'Storage_access', 'Market_access', '1.8 Gender',
       '1.9 Age', '2.2 Plot area(sqm)',
       '4.15 Has this plot been irrigated during this agricultural season?',
       '3.3 Have you used organic fertilizer in this plot during this season?',
       '3.9 Have you used inorganic fertilizer in this plot during this season?',
       '3.18 Did you use pesticide/Fungicide in any of your plots during this season?',
       '4.1 What is the degree of erosion on this plot?',
       '4.6 Is this plot located in land consolidated site in this season?',
       'District_Cl_BURERA', 'District_Cl_GAKENKE', 'District_Cl_GASABO',
       'District_Cl_GATSIBO', 'District_Cl_GICUMBI', 'District_Cl_GISAGARA',
       'District_Cl_HUYE', 'District_Cl_KAMONYI', 'District_Cl_KARONGI',
       'District_Cl_KAYONZA', 'District_Cl_KICUKIRO', 'District_Cl_KIREHE',
       'District_Cl_MUHANGA', 'District_Cl_MUSANZE', 'District_Cl_NGOMA',
       'District_Cl_NGORORERO', 'District_Cl_NYABIHU',

In [64]:
ml_data["1.8 Gender"] = ml_data["1.8 Gender"].map({
    "Male": 1,
    "Female": 0
})

In [65]:
ml_data["1.8 Gender"].unique()

array([1, 0], dtype=int64)

In [67]:
X_train = X_train.apply(pd.to_numeric, errors='coerce')
X_test = X_test.apply(pd.to_numeric, errors='coerce')

In [68]:
X_train = X_train.fillna(0)
X_test = X_test.fillna(0)

In [69]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [72]:
X = data[[ 
    "1.8 Gender",
    "1.9 Age",
    "2.2 Plot area(sqm)",
    "4.15 Has this plot been irrigated during this agricultural season?",
    "3.3 Have you used organic fertilizer in this plot during this season?",
    "3.9 Have you used inorganic fertilizer in this plot during this season?",
    "3.18 Did you use pesticide/Fungicide in any of your plots during this season?",
    "4.1 What is the degree of erosion on this plot?",
    "4.6 Is this plot located in land consolidated site in this season?",
    "District_Cl"
]]

In [73]:
X = pd.get_dummies(X, drop_first=True)

In [74]:
print(X.dtypes.value_counts())
print(X.select_dtypes(include="object").columns)

bool       38
float64     2
Name: count, dtype: int64
Index([], dtype='object')


In [79]:
print(X.select_dtypes(include="object").columns)

Index([], dtype='object')


In [81]:
print(X.select_dtypes(include="object").columns)

Index([], dtype='object')


In [84]:
from sklearn.linear_model import LogisticRegression

logit = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

logit.fit(X_train_scaled, y_train)

y_pred = logit.predict(X_test_scaled)

In [85]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

print("Accuracy:", accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.6021314387211367
[[295 207]
 [ 17  44]]
              precision    recall  f1-score   support

         0.0       0.95      0.59      0.72       502
         1.0       0.18      0.72      0.28        61

    accuracy                           0.60       563
   macro avg       0.56      0.65      0.50       563
weighted avg       0.86      0.60      0.68       563



In [86]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

ridge = LogisticRegression(
    penalty='l2',
    solver='liblinear',
    class_weight='balanced',
    max_iter=1000
)

ridge.fit(X_train_scaled, y_train)

y_pred_ridge = ridge.predict(X_test_scaled)

print("=== RIDGE RESULTS ===")
print("Accuracy:", accuracy_score(y_test, y_pred_ridge))
print(confusion_matrix(y_test, y_pred_ridge))
print(classification_report(y_test, y_pred_ridge))

=== RIDGE RESULTS ===
Accuracy: 0.6003552397868561
[[294 208]
 [ 17  44]]
              precision    recall  f1-score   support

         0.0       0.95      0.59      0.72       502
         1.0       0.17      0.72      0.28        61

    accuracy                           0.60       563
   macro avg       0.56      0.65      0.50       563
weighted avg       0.86      0.60      0.68       563



In [87]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

lasso = LogisticRegression(
    penalty='l1',
    solver='liblinear',
    class_weight='balanced',
    max_iter=1000
)

lasso.fit(X_train_scaled, y_train)

y_pred_lasso = lasso.predict(X_test_scaled)

print("=== LASSO RESULTS ===")
print("Accuracy:", accuracy_score(y_test, y_pred_lasso))
print(confusion_matrix(y_test, y_pred_lasso))
print(classification_report(y_test, y_pred_lasso))

=== LASSO RESULTS ===
Accuracy: 0.6003552397868561
[[295 207]
 [ 18  43]]
              precision    recall  f1-score   support

         0.0       0.94      0.59      0.72       502
         1.0       0.17      0.70      0.28        61

    accuracy                           0.60       563
   macro avg       0.56      0.65      0.50       563
weighted avg       0.86      0.60      0.68       563



In [89]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [90]:
rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced"
)

rf.fit(X_train_scaled, y_train)

y_pred_rf = rf.predict(X_test_scaled)

print("=== RANDOM FOREST RESULTS ===")
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print(confusion_matrix(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

=== RANDOM FOREST RESULTS ===
Accuracy: 0.872113676731794
[[474  28]
 [ 44  17]]
              precision    recall  f1-score   support

         0.0       0.92      0.94      0.93       502
         1.0       0.38      0.28      0.32        61

    accuracy                           0.87       563
   macro avg       0.65      0.61      0.63       563
weighted avg       0.86      0.87      0.86       563



In [93]:
y_reg = data["PHL_rate"]

In [94]:
from sklearn.model_selection import train_test_split

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X, y_reg,
    test_size=0.2,
    random_state=42
)

In [95]:
from sklearn.ensemble import RandomForestRegressor

rf_reg = RandomForestRegressor(
    n_estimators=500,
    max_depth=10,
    min_samples_leaf=2,
    random_state=42
)

rf_reg.fit(X_train_reg, y_train_reg)

y_pred_reg = rf_reg.predict(X_test_reg)

In [96]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

rmse = np.sqrt(mean_squared_error(y_test_reg, y_pred_reg))
mae = mean_absolute_error(y_test_reg, y_pred_reg)
r2 = r2_score(y_test_reg, y_pred_reg)

print("=== RANDOM FOREST REGRESSION RESULTS ===")
print("R²   :", r2)
print("RMSE :", rmse)
print("MAE  :", mae)

=== RANDOM FOREST REGRESSION RESULTS ===
R²   : 0.06189935837336713
RMSE : 0.079925411131868
MAE  : 0.020077744090183543


In [97]:
class_weight="balanced"

In [99]:
y_proba = rf.predict_proba(X_test)[:, 1]

In [100]:
y_proba = logit.predict_proba(X_test)[:, 1]

In [101]:
y_proba = ridge.predict_proba(X_test)[:, 1]

In [102]:
y_proba = rf.predict_proba(X_test)[:, 1]

threshold = 0.3
y_pred = (y_proba >= threshold).astype(int)

In [103]:
print(rf)
print(logit)
print(ridge)
print(lasso)

RandomForestClassifier(class_weight='balanced', n_estimators=300,
                       random_state=42)
LogisticRegression(class_weight='balanced', max_iter=1000)
LogisticRegression(class_weight='balanced', max_iter=1000, solver='liblinear')
LogisticRegression(class_weight='balanced', max_iter=1000, penalty='l1',
                   solver='liblinear')
